In [1]:
import json
import math
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------
CLUB_NAME = "Real Madrid"
TOP_N = 5  # how many metrics to list in each strengths/weaknesses sentence

# Metrics where "higher is better"
HIGHER_BETTER = {
    "goals", "xg", "shots", "shots_on_target", "box_touches",
    "ball_possession_pct", "field_tilt_pct", "win_probability_pct",
    "points", "goal_difference", "recoveries", "recoveries_within_5s_pct",
    "np_xg_per_shot", "defensive_duels_won_pct",
    "defensive_action_height_m", "recovery_line_height_m", "turnover_line_height_m",
    "final_third_to_box_pct", "box_to_shot_pct", "possessions_to_final_third_pct",
    "xpts", "xt", "xg_within_10s_after_recovery", "xt_within_10s_after_recovery",
    "possessions_to_box_within_10s_after_recovery", "possessions_to_final_third_within_10s_after_recovery"
}

# Metrics where "lower is better" (primarily opposition outputs, fouls/cards, time/PPDA)
LOWER_BETTER = {
    "opp_goals", "opp_xg", "opp_np_xg", "opp_shots", "opp_np_shots",
    "opp_box_touches", "opp_np_xg_per_shot", "opp_high_opportunity_shots",
    "ppda", "fouls_commited", "yellow_cards", "red_cards",
    "own_goals", "offsides", "time_to_defensive_action_s", "time_to_recovery_s"
}

# You can expand HIGHER_BETTER / LOWER_BETTER if other columns in your file are relevant.


# -----------------------------
# Helper functions
# -----------------------------
def percent_rank(series, higher_is_better=True):
    """
    Compute percentile rank (0..1) for each value in a pandas Series.
    If higher_is_better=False, invert so that lower values get higher percentiles.
    """
    # Handle constant series case
    if series.nunique(dropna=True) <= 1:
        return pd.Series([0.5] * len(series), index=series.index)

    # Rank from low to high, ties average
    r = series.rank(method="average", na_option="keep")
    n = r.notna().sum()
    pr = (r - 1) / (n - 1)
    if not higher_is_better:
        pr = 1 - pr
    return pr


def fmt_val(v):
    """Pretty numeric formatting."""
    if pd.isna(v):
        return "–"
    # Choose decimals based on magnitude
    if abs(v) >= 100:
        return f"{v:,.0f}"
    if abs(v) >= 10:
        return f"{v:,.2f}"
    return f"{v:,.3f}"


def join_metric_phrases(rows, metric_map, sep=", ", max_items=TOP_N):
    """
    rows: list of tuples (pretty_name, value, percentile_float)
    Returns a human-readable inline list: "<Name> (value, pXX), ..."
    """
    out = []
    for name, val, p in rows[:max_items]:
        pretty = metric_map.get(name, name)
        # percentile to nearest integer (e.g., 0.874 -> 87th)
        pct = int(round(p * 100))
        # Add superscript for ordinal? Keep simple: "87th pct"
        out.append(f"{pretty} ({fmt_val(val)}, {pct}th pct)")
    return sep.join(out)


# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv("/data/demo_data/football_test/team_stats.csv")
with open("/data/demo_data/football_test/match_api_metric_map.json", "r") as f:
    metric_map = json.load(f)

# Confirm the club exists
if CLUB_NAME not in set(df["club_name"].unique()):
    raise ValueError(f"Club '{CLUB_NAME}' not found in the data.")

# Club row and cohort (same league & season for fair comparison)
club_row = df[df["club_name"] == CLUB_NAME].iloc[0]
cohort = df[(df["competition_id"] == club_row["competition_id"]) &
            (df["season"] == club_row["season"])].copy()

# Intersect configured metrics with columns present in the file
available_higher = [c for c in HIGHER_BETTER if c in cohort.columns]
available_lower = [c for c in LOWER_BETTER if c in cohort.columns]

# Compute percentile ranks for both sets
strength_records = []  # (metric, value, percentile)
for c in available_higher:
    pr = percent_rank(cohort[c], higher_is_better=True)
    strength_records.append((c, club_row[c], float(pr.loc[club_row.name])))

for c in available_lower:
    pr = percent_rank(cohort[c], higher_is_better=False)  # invert logic
    strength_records.append((c, club_row[c], float(pr.loc[club_row.name])))

# Filter out non-numeric or NaN percentiles
strength_records = [(m, v, p) for (m, v, p) in strength_records if isinstance(p, float) and not math.isnan(p)]

# Sort by percentile descending to find top strengths
strength_records_sorted = sorted(strength_records, key=lambda x: x[2], reverse=True)

# Also find potential weaknesses (lowest percentiles)
weakness_records_sorted = sorted(strength_records, key=lambda x: x[2])

# Curate strengths: prefer attacking/positive metrics first
# Strategy: take top-N, but ensure diversity by favoring HIGHER_BETTER metrics when present
top_strengths = []
for m, v, p in strength_records_sorted:
    # Skip if not in file (already ensured), just collect best
    top_strengths.append((m, v, p))
    if len(top_strengths) >= TOP_N:
        break

# Curate weaknesses: take bottom-N, but avoid purely opponent-output duplicates if possible
bottom_weaknesses = []
for m, v, p in weakness_records_sorted:
    bottom_weaknesses.append((m, v, p))
    if len(bottom_weaknesses) >= TOP_N:
        break

# Build sentence fragments
strengths_inline = join_metric_phrases(top_strengths, metric_map, max_items=TOP_N)
weaknesses_inline = join_metric_phrases(bottom_weaknesses, metric_map, max_items=TOP_N)

# -----------------------------
# Compose the three sentences
# -----------------------------
strength_sentence = (
    f"Real Madrid’s clearest strengths emerge in the metrics where they rate highest—"
    f"{strengths_inline}."
)

weakness_sentence = (
    f"Areas that are closer to average or relatively weaker include "
    f"{weaknesses_inline}."
)

conclusion_sentence = (
    "Overall, Real Madrid profile as an assertive, chance‑creating side whose attacking volume and quality outweigh a few defensive or transitional inefficiencies."
)

# -----------------------------
# Output
# -----------------------------
print(strength_sentence)
print(weakness_sentence)
print(conclusion_sentence)

Real Madrid’s clearest strengths emerge in the metrics where they rate highest—Possessions to final third % (0.611, 100th pct), xG within 10s after recovery (0.287, 100th pct), Turnover line height (m) (66.48, 100th pct), Yellow cards (1.526, 100th pct), Fouls committed (8.711, 100th pct).
Areas that are closer to average or relatively weaker include Recoveries within 5s % (0.221, 5th pct), Time to recovery (s) (12.02, 11th pct), Offsides (2.105, 13th pct), Box to shot % (0.666, 21th pct), Time to defensive action (s) (6.485, 32th pct).
Overall, Real Madrid profile as an assertive, chance‑creating side whose attacking volume and quality outweigh a few defensive or transitional inefficiencies.
